In [1]:
import pandas as pd

In [2]:
# chargement des csv
transactions = pd.read_csv('data/raw/transactions.csv')
clients = pd.read_csv('data/raw/clients.csv')

In [3]:
# Nombre de lignes et colonnes
print(transactions.shape)

(15, 8)


In [4]:
# Types de chaque colonne
print(transactions.dtypes)

transaction_id     object
client_id          object
date               object
montant           float64
type               object
categorie          object
statut             object
ville              object
dtype: object


In [5]:
# 5 premières lignes
print(transactions.head(5))

  transaction_id client_id        date  montant      type     categorie  \
0           T001       C01  2024-01-05    150.0     achat  alimentation   
1           T002       C02  2024-01-07    820.5     achat  électronique   
2           T003       C03  2024-01-08     45.0  virement           NaN   
3           T004       C01  2024-01-10    310.0     achat        voyage   
4           T005       C04  2024-01-12      NaN     achat  alimentation   

       statut      ville  
0      validé      Paris  
1      validé       Lyon  
2      validé  Marseille  
3  en attente      Paris  
4      validé   Bordeaux  


In [6]:
# Nombre de valeurs nulles par colonne
print(transactions.isnull().sum())

transaction_id    0
client_id         0
date              0
montant           3
type              1
categorie         1
statut            0
ville             4
dtype: int64


### Filtre

In [7]:
# Uniquement les transactions de type 'achat'
transactions_achat = transactions[transactions['type']=='achat']
transactions_achat

,transaction_id,client_id,date,montant,type,categorie,statut,ville
0,T001,C01,2024-01-05,150.0,achat,alimentation,validé,Paris
1,T002,C02,2024-01-07,820.5,achat,électronique,validé,Lyon
3,T004,C01,2024-01-10,310.0,achat,voyage,en attente,Paris
4,T005,C04,2024-01-12,NaN,achat,alimentation,validé,Bordeaux
5,T006,C05,2024-01-15,95.0,achat,Électronique,validé,NaN
6,T007,C02,2024-01-18,1200.0,achat,voyage,validé,Lyon
8,T009,C07,2024-01-22,560.0,achat,électronique,en attente,NaN
10,T011,C08,2024-01-28,75.0,achat,alimentation,validé,Strasbourg
11,T012,C01,2024-02-01,430.0,achat,voyage,validé,Paris
12,T013,C09,2024-02-03,88.0,achat,alimentation,validé,Nantes


In [8]:
# Uniquement les colonnes : client_id, montant, categorie
transactions_cols = transactions[['client_id', 'montant', 'categorie']]
transactions_cols

,client_id,montant,categorie
0,C01,150.0,alimentation
1,C02,820.5,électronique
2,C03,45.0,NaN
3,C01,310.0,voyage
4,C04,NaN,alimentation
5,C05,95.0,Électronique
6,C02,1200.0,voyage
7,C06,30.0,alimentation
8,C07,560.0,électronique
9,C03,NaN,rejeté


In [9]:
# Les transactions où montant > 200 ET statut == 'validé'
transactions_filtre = transactions[(transactions['montant'] > 200) & (transactions['statut']=='validé')]
transactions_filtre

,transaction_id,client_id,date,montant,type,categorie,statut,ville
1,T002,C02,2024-01-07,820.5,achat,électronique,validé,Lyon
6,T007,C02,2024-01-18,1200.0,achat,voyage,validé,Lyon
11,T012,C01,2024-02-01,430.0,achat,voyage,validé,Paris
14,T015,C04,2024-02-08,640.0,achat,électronique,validé,Bordeaux


### Calcule par categorie :

In [10]:
# Total des montants
total_montants = transactions.groupby('categorie')['montant'].sum()
print(total_montants)

categorie
 Voyage              0.0
 alimentation       75.0
 Électronique       95.0
alimentation       268.0
rejeté               0.0
voyage            1940.0
électronique      2020.5
Name: montant, dtype: float64


In [11]:
# Moyenne des montants
moyenne_montants = transactions.groupby('categorie')['montant'].mean()
print(moyenne_montants.round(2))

categorie
 Voyage              NaN
 alimentation      75.00
 Électronique      95.00
alimentation       89.33
rejeté               NaN
voyage            646.67
électronique      673.50
Name: montant, dtype: float64


In [12]:
# Nombre de transactions
nb_transactions = transactions.groupby('categorie')['transaction_id'].count()
print(nb_transactions)

categorie
 Voyage           1
 alimentation     1
 Électronique     1
alimentation      4
rejeté            1
voyage            3
électronique      3
Name: transaction_id, dtype: int64


In [13]:
calcule = transactions.groupby('categorie').agg(
    total_montants = ('montant', 'sum'),
    moyenne_montants = ('montant', 'mean'),
    nb_transactions = ('categorie', 'count')
)
print(calcule.reset_index())

        categorie  total_montants  moyenne_montants  nb_transactions
0         Voyage              0.0               NaN                1
1    alimentation            75.0         75.000000                1
2   Électronique             95.0         95.000000                1
3    alimentation           268.0         89.333333                4
4          rejeté             0.0               NaN                1
5          voyage          1940.0        646.666667                3
6    électronique          2020.5        673.500000                3


## Jointure

In [14]:
transactions_clients_inner = pd.merge(transactions, clients, on='client_id', how='inner')

In [15]:
transactions_clients_left = pd.merge(transactions, clients, on='client_id', how='left')

In [16]:
transactions_clients_left[transactions_clients_left['nom'].isnull()]

,transaction_id,client_id,date,montant,type,categorie,statut,ville_x,nom,age,ville_y,segment
13,T014,C10,2024-02-05,NaN,achat,Voyage,en attente,NaN,NaN,NaN,NaN,NaN


### Nettoyage des valeurs nulls

In [17]:
# Supprissions des lignes avec plus de 3 valeurs nulles
transactions_clean = transactions.dropna(thresh=5)

In [18]:
# Remplacer  les montants nulls par la médiane
transactions_clean['montant'] = transactions_clean['montant'].fillna(transactions_clean['montant'].median())

In [19]:
transactions_clean['ville'] = transactions_clean['ville'].fillna('Inconnu')

### Créations de nouvelles colonnes

In [20]:
transactions_clean['frais'] = (transactions_clean['montant'] * 0.015).round(2)
transactions_clean['montant_net'] = transactions_clean['montant'] - transactions_clean['frais']

transactions_clean['niveau'] = transactions_clean['montant'].apply(
    lambda x : 'premium' if x > 500 else 'standard'
)

# Normalisation
transactions_clean['categorie'] = transactions_clean['categorie'].str.strip().str.lower()

# Converssion en datetime
transactions_clean['date'] = pd.to_datetime(transactions_clean['date'])


### Export 

In [21]:
transactions_clean.to_csv('data/export/transactions_clean.csv', index=False)
transactions_clean[transactions_clean['niveau']=='premium'].to_excel('data/export/transactions_premium.xlsx', index=False,  sheet_name='premium')
transactions_clean.to_json('data/export/transactions_cleans.json', orient='records', indent=2, force_ascii=False)

print("✅ Exports terminés")

✅ Exports terminés
